
# Parte 3 - Entrenamiento de una Red Neuronal (MLP)

## Detección de objetos peligrosos

En este notebook se implementa una red neuronal tipo perceptrón Multicapa (MLP) para clasificar imágenes en las siguientes categorías:

- alcohol
- blood
- cigarette
- gun
- insulting_gesture
- knife


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image



## 1. Carga del dataset

El dataset debe estar organizado y cada carpeta debe contener imágenes de su respectiva categoría.

en este caso se transdorman las imagenes a 64 x 64 px para que el procesamiento sea un poco mas rapido, y se use menos memoria 


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Ruta del dataset
dataset_path = '../data/class'


# Cargar imagenes
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

print("Cantidad total de imágenes:", len(dataset))
print("Clases encontradas:", dataset.classes)



## 3. División de entrenamiento y validación

En este punto se determina la cantidad de imagenes con las cuales la red aprendera y cuales seran para validacion, se ordenan de manera aleatoria para el correcto aprendizaje del mismo 

In [ ]:

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Imágenes entrenamiento:", len(train_dataset))
print("Imágenes validación:", len(val_dataset))



## 4. Visualización de ejemplos


In [ ]:

classes = dataset.classes

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(1, 5, figsize=(15,5))

for i in range(5):
    img = images[i].permute(1,2,0).numpy()
    img = (img * 0.5) + 0.5

    axes[i].imshow(img)
    axes[i].set_title(classes[labels[i]])
    axes[i].axis("off")

plt.show()



## 5. Construcción del modelo MLP

Para crea el modelo de perceptrón Multicapa, las imágenes se aplanan en un vector y luego pasan por varias capas densas


In [ ]:

class DangerousObjectMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 64 * 3, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 6)
        )

    def forward(self, x):
        return self.network(x)

model = DangerousObjectMLP().to(device)

print(model)


## 6. Función de pérdida y optimizador


In [ ]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)



## 7. Entrenamiento de la red neuronal


In [ ]:

epochs = 15

train_losses = []
val_accuracies = []

for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Reiniciar gradientes
        optimizer.zero_grad()

        # Forward
        outputs = model(images)

        # Loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Actualización de pesos
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Evaluación
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    val_accuracies.append(accuracy)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("-" * 40)



## 8. Gráficas de entrenamiento


In [ ]:

plt.figure(figsize=(10,4))

plt.plot(train_losses)
plt.title("Pérdida de entrenamiento")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.show()


In [ ]:

plt.figure(figsize=(10,4))

plt.plot(val_accuracies)
plt.title("Precisión de validación")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.show()



## 9. Guardar el modelo entrenado


In [ ]:

torch.save(model.state_dict(), "dangerous_objects_mlp.pth")

print("Modelo guardado correctamente.")



## 10. Predicción sobre una imagen nueva


In [ ]:

def predict_image(image_path):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    predicted_class = classes[predicted.item()]

    return predicted_class


In [ ]:

#Agregar ruta de la imagen a predecir

image_path = '../data/class/pistola.jpg'

predicted = predict_image(image_path)
print("Predicción:", predicted)



# Conclusiones

- Se implementó una red neuronal tipo MLP para clasificar objetos peligrosos.
- El modelo utiliza imágenes redimensionadas a 64x64 píxeles.
- Se empleó PyTorch para el entrenamiento y evaluación.
- La función de activación utilizada fue ReLU.
- Se utilizó Adam como optimizador.
- El modelo puede identificar las categorías:
  alcohol, blood, cigarette, gun, insulting_gesture y knife.
